In [47]:
MATCH_PROMPT = """
You are an expert job matching and recruitment assistant.

IMPORTANT TERMINOLOGY:
- "You" refers ONLY to the person described in "YOUR PROFESSIONAL SUMMARY".
- "You" does NOT refer to the AI assistant.
- Always use "you" and "your" when describing the person's experience.
- Never refer to the person as "the candidate", "the applicant", or "they".

Your task is to determine whether you are a suitable match for the job.

DECISION RULES
==============

There are exactly three possible decisions:

1. "direct_apply"

Use "direct_apply" when you have most of the important skills,
experience, and qualifications required for the job.

Minor missing skills are acceptable.

2. "needs_clarification"

Use "needs_clarification" when important information is missing
or unclear and that information could materially change the decision.

Examples:
- Required years of experience are unknown.
- Work authorization is unknown.
- A mandatory certification is mentioned but your possession
  of it is unknown.
- An important job requirement is unclear.

3. "reject"

Use "reject" only when you clearly fail a critical requirement.

Examples:
- Major skills mismatch.
- Required experience is substantially different.
- A mandatory certification is explicitly absent.
- A mandatory degree is explicitly absent.
- A mandatory technology is explicitly absent.
- The role is clearly incompatible with your profile.


MATCHING RULES
==============

- Do not reject you because a skill is simply not mentioned.
- "Not mentioned" is different from "explicitly absent".
- Missing information should normally result in "needs_clarification".
- Mandatory requirements are more important than preferred requirements.
- Consider transferable skills.
- Do not require a 100% keyword match.
- Consider semantic similarity.
- Consider seniority and experience.
- Consider technologies, frameworks, tools, responsibilities,
  education, certifications, and domain experience.
- Do not invent your experience or qualifications.
- Do not invent job requirements.
- Do not assume an unmentioned skill is absent.


REQUIREMENTS VS RESPONSIBILITIES
================================

Do NOT treat every sentence in the job description as a qualification.

For example:

"Work closely with backend engineers"

does NOT mean:

"You must have backend engineering experience."

Similarly:

"Work closely with designers"

does NOT mean:

"You must have experience as a designer."

Only treat something as a missing requirement when the job explicitly
requires it or strongly indicates that it is a qualification.

Do not put ordinary job responsibilities into "missing_or_unclear".


MATCH SCORE
===========

Return an integer from 0 to 100.

90-100 = Excellent match
75-89  = Strong match
60-74  = Moderate or uncertain match
40-59  = Weak match
0-39   = Very poor match

The score must reflect the overall fit.

Do not use the score alone to determine the decision.
Critical mandatory requirements take priority.

REASON
======

Explain the decision using concrete evidence from the professional
summary and job object.

Do not give a generic reason such as:
"Most of the required skills are present."

Mention the most important matching skills or experience and explain
why they support the decision.

Keep the reason concise: 2 to 3 sentences.

MATCHING SKILLS
===============

Include only skills, qualifications, experience, technologies,
or capabilities that clearly match the job.


MISSING OR UNCLEAR
==================

Include only actual requirements that are:
- missing from your professional summary, or
- unclear from the available information.
- Only list a missing item if it is explicitly stated or strongly implied
  as a qualification or requirement by the job.
- Do not list a skill merely because it appears in the job description.
- Do not treat tools, technologies, or activities mentioned only in the
  description of responsibilities as required qualifications unless the
  job clearly presents them as requirements.


Do not create missing requirements from ordinary job responsibilities.


CRITICAL GAPS
=============

Include only critical mandatory requirements that you clearly do not satisfy.

Do not put ordinary missing skills here.


FUTURE WORK EXPERIENCE
======================

Write 3 to 4 concise lines describing the type of work you would
likely perform if selected.

Base this only on the job and your demonstrated capabilities.

Do not invent unrelated responsibilities.


YOUR PROFESSIONAL SUMMARY
=========================

{user_summary}


JOB OBJECT
==========

{job_object}


ADDITIONAL USER INSTRUCTION
===========================

{user_instruction}


FINAL OUTPUT REQUIREMENT
========================

{format_instructions}

CRITICAL:

Return ONLY the structured output described above.

DO NOT:
- explain your reasoning
- write an introduction
- write a conclusion
- use Markdown
- use bullet points
- use headings
- write "Match Result:"
- write "Match Score:"
- write "Reason:"
- add any text before the structured output
- add any text after the structured output

Your entire response must contain ONLY the required structured output.
"""


In [48]:
from enum import Enum
from pydantic import BaseModel, Field


class MatchResult(str, Enum):
    DIRECT_APPLY = "direct_apply"
    NEEDS_CLARIFICATION = "needs_clarification"
    REJECT = "reject"


class JobMatchResult(BaseModel):
    result: MatchResult = Field(
        description="Final job matching decision"
    )

    match_score: int = Field(
        ge=0,
        le=100,
        description="Overall match score from 0 to 100"
    )

    reason: str = Field(
        description=(
            "Explain why the decision was made using the most important "
            "matching skills, experience, and any meaningful gaps. "
            "Do not give a generic statement."
        )
    )

    matching_skills: list[str] = Field(
        default_factory=list,
        description="Skills and qualifications matching the job"
    )

    missing_or_unclear: list[str] = Field(
        default_factory=list,
        description="Information that is missing or unclear from the professional summary"

    )

    critical_gaps: list[str] = Field(
        default_factory=list,
        description="Critical requirements that the candidate does not satisfy"
    )

    future_work_experience: str = Field(
        description=(
            "A concise 3 to 4 line explanation of the type of work "
            "the candidate would likely perform if selected for this job"
        )
    )

In [49]:
import os
import json
from enum import Enum

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError

from langchain_huggingface import (
    HuggingFaceEndpoint,
    ChatHuggingFace,
)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import OutputFixingParser

# from .match_job_prompt import MATCH_PROMPT
# from .match_job_schema import JobMatchResult


load_dotenv()

# ---------------------------------------------------------
# Model
# ---------------------------------------------------------

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    max_new_tokens=1024,
    temperature=0.1,
)

model = ChatHuggingFace(llm=llm)


# =========================================================
# Pydantic Output Parser
# =========================================================

parser = PydanticOutputParser(
    pydantic_object=JobMatchResult
)


# =========================================================
# Output Fixing Parser
# =========================================================

fixing_parser = OutputFixingParser.from_llm(
    parser=parser,
    llm=model,
    max_retries=2,
)


# ---------------------------------------------------------
# Prompt
# ---------------------------------------------------------

prompt = ChatPromptTemplate.from_template(MATCH_PROMPT)

# =========================================================
# Chain
# =========================================================

match_chain = prompt | model | fixing_parser

# ---------------------------------------------------------
# Function
# ---------------------------------------------------------

def match_user_to_job(
    user_summary: str,
    job: dict,
    user_instruction: str = "",
) -> JobMatchResult:

    if not user_summary or not user_summary.strip():
        raise ValueError("user_summary must not be empty")

    if not job:
        raise ValueError("job must not be empty")

    try:

        result = match_chain.invoke({
            "user_summary": user_summary.strip(),

            "job_object": json.dumps(
                job,
                indent=2,
                ensure_ascii=False,
            ),

            "user_instruction": (
                user_instruction.strip()
                if user_instruction
                else "No additional instruction."
            ),

            "format_instructions": (
                parser.get_format_instructions()
            ),
        })

        print(result.result.value)
        print(result.match_score)
        print(result.reason)
        print(result.matching_skills)
        print(result.missing_or_unclear)
        print(result.critical_gaps)
        print(result.future_work_experience)

        return result

    except OutputParserException as e:

        print("Output parser failed:")
        print(e)

        raise

    except ValidationError as e:

        print("Pydantic validation failed:")
        print(e)

        raise

    except Exception as e:

        print("Failed to parse model output:")
        print(e)

        raise

In [50]:
summary = """
Here is the final output based on the provided source content:

**Professional Representation of Anirban Das**

**Core Capabilities**

* **Software Engineering**: Experienced in developing scalable and efficient software systems using Java, TypeScript, and C++.
* **Automation**: Skilled in designing and implementing automation systems using FastAPI, Next.js, and LangChain.
* **Collaborative Development**: Proficient in building real-time collaborative code editors using Yjs and Monaco Editor.
* **AI/ML**: Familiar with AI/ML tools such as LangChain, LangGraph, TensorFlow, and Keras.
* **Database Management**: Experienced in designing and implementing database systems using MongoDB and PostgreSQL.

**Technical Expertise**

* **Programming Languages**: Proficient in C++, Java, TypeScript, JavaScript, and Python.
* **Frameworks**: Experienced in using React.js, Next.js, Node.js, Express.js, and FastAPI.
* **Databases**: Familiar with MongoDB, PostgreSQL, and Vector Databases.
* **AI/ML Tools**: Skilled in using LangChain, LangGraph, TensorFlow, and Keras.
* **Automation Tools**: Experienced in using FastAPI, Next.js, and Socket.IO.

**Domain Experience**

* **Software Engineering**: Experienced in developing software systems for Google and other clients.
* **Automation**: Skilled in designing and implementing automation systems for various industries.
* **Collaborative Development**: Proficient in building real-time collaborative code editors for distributed teams.
* **AI/ML**: Familiar with AI/ML tools and techniques for various applications.
* **Database Management**: Experienced in designing and implementing database systems for various industries.

**Professional Experience**

* **Software Engineer Intern**, Google (May 2026 - July 2026)
	+ Engineered a new linting rule and automated batch validation pipeline in Java and TypeScript.
	+ Enhanced core template management interfaces by developing a responsive document comparison dialog box.
	+ Contributed 7,350+ lines of production code across 22 peer-reviewed changelists.
* **Automation System Developer**, JobPilot (GitHub)
	+ Developed an automation system using FastAPI, Next.js, and HuggingFace LLMs.
	+ Architected a scalable asynchronous worker system with WebSockets/Socket.IO updates.
	+ Built an LLM-powered job ranking and categorization pipeline using prompt engineering.

**Projects**

* **Code Fusion**: A real-time collaborative code editor using Yjs and Monaco Editor.
* **JobPilot**: An automation system using FastAPI, Next.js, and LangChain.
* **NoteBridge**: A feature-rich note-taking and sharing platform.

**Achievements**

* **4th rank at HaXplore | CodeFest'25**, organized by IIT BHU.
* **Winner** of Winter of Code 6.0 (Web Development Division), a one-month long hackathon conducted by CyberLabs, IIT (ISM) Dhanbad.

**Education**

* **Bachelor of Technology in Computer Science and Engineering**, Indian Institute of Technology (ISM), Dhanbad (2023 - 2027).

**Social Engagements**

* **Member of CyberLabs -Tech society of IIT ISM Dhanbad**.
* **Member of Aquatics Team - Swimming Team of IIT ISM Dhanbad**.
* **Represented IIT Dhanbad at the 37th INTER IIT AQUATICS MEET 2023 held at IIT Gandhinagar and secured 4th place in 200m Individual Medley**.

This output represents the user's professional identity and capabilities as completely as possible, preserving important specific terminology from the source, broader professional concepts when clearly supported, measurable evidence, and context. The output is optimized for semantic comparison with future job descriptions.
"""

In [51]:
job = {
    "id": "job-101",
    "title": "Frontend Developer",
    "company": "TechCorp",
    "cities": ["Chennai", "Bangalore"],
    "countries": ["India"],
    "is_remote": True,
    "is_hybride": True,
    "is_onsite": False,
    "salary_offered": "₹8,00,000 - ₹14,00,000 per year",
    "visa_sponsorship_offered": False,
    "start_date": "2026-09-15",
    "required_skills": [
      "React",
      "TypeScript",
      "JavaScript",
      "HTML",
      "CSS",
      "Git"
    ],
    "description": "Build responsive customer-facing interfaces using React and TypeScript. Work closely with designers and backend engineers to create reusable components and improve frontend performance."
  }

In [52]:
match_user_to_job(user_summary=summary , job=job)

needs_clarification
85
You have experience in developing scalable and efficient software systems using Java, TypeScript, and C++. However, the job requires React and TypeScript skills, which you have demonstrated proficiency in. Additionally, the job requires work experience in Chennai or Bangalore, which is unclear from your professional summary.
['React', 'TypeScript', 'JavaScript', 'HTML', 'CSS', 'Git']
['Work experience in Chennai or Bangalore']
[]
As a Frontend Developer, you would likely perform tasks such as building responsive customer-facing interfaces using React and TypeScript, working closely with designers and backend engineers to create reusable components, and improving frontend performance.


JobMatchResult(result=<MatchResult.NEEDS_CLARIFICATION: 'needs_clarification'>, match_score=85, reason='You have experience in developing scalable and efficient software systems using Java, TypeScript, and C++. However, the job requires React and TypeScript skills, which you have demonstrated proficiency in. Additionally, the job requires work experience in Chennai or Bangalore, which is unclear from your professional summary.', matching_skills=['React', 'TypeScript', 'JavaScript', 'HTML', 'CSS', 'Git'], missing_or_unclear=['Work experience in Chennai or Bangalore'], critical_gaps=[], future_work_experience='As a Frontend Developer, you would likely perform tasks such as building responsive customer-facing interfaces using React and TypeScript, working closely with designers and backend engineers to create reusable components, and improving frontend performance.')